Ce codelab s'appuie largement sur un [exemple de code de scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html).

In [ ]:
import warnings

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
%matplotlib inline

# Forêts aléatoires

Le module sklearn.ensemble contient deux algorithmes de moyennage fondés sur des arbres de décision aléatoires : l'algorithme RandomForest et la méthode Extra-Trees. Ce sont deux techniques de type « perturber puis combiner » conçues spécifiquement pour les arbres : on crée un ensemble varié de classifieurs en introduisant de l'aléa dans leur construction. La prédiction de l'ensemble est la moyenne des prédictions des classifieurs individuels.

Comme les autres classifieurs, les forêts s'entraînent avec deux tableaux : un tableau X, creux ou dense, de taille [n_samples, n_features], qui contient les exemples d'entraînement, et un tableau Y de taille [n_samples], qui contient les valeurs cibles (labels de classe) des exemples d'entraînement :

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = [[0, 0], [1, 1]]
Y = [0, 1]
model = RandomForestClassifier(n_estimators=10)
model = model.fit(X, Y)
print(model)

Dans une forêt aléatoire (voir les classes RandomForestClassifier et RandomForestRegressor), chaque arbre de l'ensemble est construit à partir d'un échantillon tiré avec remise (un échantillon *bootstrap*) dans le jeu d'entraînement. De plus, quand on coupe un nœud pendant la construction de l'arbre, la coupure choisie n'est plus la meilleure parmi toutes les variables, mais la meilleure parmi un sous-ensemble aléatoire de variables. Cet aléa augmente en général légèrement le biais de la forêt (par rapport à celui d'un arbre unique non aléatoire), mais le moyennage réduit aussi sa variance, en général bien plus que ne croît le biais : on obtient donc globalement un meilleur modèle.

Contrairement à la publication originale, l'implémentation de scikit-learn combine les classifieurs en moyennant leurs prédictions probabilistes, au lieu de faire voter chaque classifieur pour une seule classe.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier

X, y = make_classification(
  n_samples=1000,
  n_features=4,
  n_informative=2,
  n_redundant=0,
  random_state=0,
  shuffle=False,
)
model = RandomForestClassifier(max_depth=2, random_state=0)
model.fit(X, y)

print(model.feature_importances_)
print(model.predict([[0, 0, 0, 0]]))

## Erreurs OOB des forêts aléatoires

Le RandomForestClassifier est entraîné par *bootstrap aggregation* : chaque nouvel arbre est entraîné sur un échantillon bootstrap des observations d'entraînement z_i = (x_i, y_i). L'erreur *out-of-bag* (OOB) est l'erreur moyenne sur chaque z_i, calculée avec les prédictions des arbres dont l'échantillon bootstrap ne contient pas z_i. Le RandomForestClassifier peut ainsi être validé pendant son entraînement.

L'exemple ci-dessous montre comment mesurer l'erreur OOB à chaque ajout d'un arbre pendant l'entraînement. Le graphique obtenu permet d'estimer une valeur de n_estimators à partir de laquelle l'erreur se stabilise.

In [ ]:
from collections import OrderedDict

import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier

# Auteurs : Kian Ho <hui.kian.ho@gmail.com>
#           Gilles Louppe <g.louppe@gmail.com>
#           Andreas Mueller <amueller@ais.uni-bonn.de>
#
# Licence : BSD 3 Clause

print(__doc__)

RANDOM_STATE = 123

# Génération d'un jeu de données de classification binaire.
X, y = make_classification(
  n_samples=500,
  n_features=25,
  n_clusters_per_class=1,
  n_informative=15,
  random_state=RANDOM_STATE,
)

# NB : passer le paramètre `warm_start` à `True` désactive la parallélisation
# des ensembles, mais c'est nécessaire pour suivre l'évolution de l'erreur OOB
# pendant l'entraînement.
ensemble_models = [
  (
    "RandomForestClassifier, max_features='sqrt'",
    RandomForestClassifier(
      warm_start=True,
      oob_score=True,
      max_features="sqrt",
      random_state=RANDOM_STATE,
    ),
  ),
  (
    "RandomForestClassifier, max_features='log2'",
    RandomForestClassifier(
      warm_start=True,
      max_features="log2",
      oob_score=True,
      random_state=RANDOM_STATE,
    ),
  ),
  (
    "RandomForestClassifier, max_features=None",
    RandomForestClassifier(
      warm_start=True,
      max_features=None,
      oob_score=True,
      random_state=RANDOM_STATE,
    ),
  ),
]

# Associe à chaque nom de classifieur une liste de paires (<n_estimators>, <taux d'erreur>).
error_rate = OrderedDict((label, []) for label, _ in ensemble_models)

# Plage de valeurs de `n_estimators` à explorer.
min_estimators = 15
max_estimators = 175

for label, model in ensemble_models:
  for i in range(min_estimators, max_estimators + 1):
    model.set_params(n_estimators=i)
    model.fit(X, y)

    # Enregistre l'erreur OOB pour chaque valeur `n_estimators=i`.
    oob_error = 1 - model.oob_score_
    error_rate[label].append((i, oob_error))

# Graphique du « taux d'erreur OOB » en fonction de « n_estimators ».
for label, model_err in error_rate.items():
  xs, ys = zip(*model_err)
  plt.plot(xs, ys, label=label)

plt.xlim(min_estimators, max_estimators)
plt.xlabel("n_estimators")
plt.ylabel("Taux d'erreur OOB")
plt.legend(loc="upper right")
plt.show()

## Affichage du premier arbre de décision grâce à l'outil GraphViz de sklearn

In [ ]:
from sklearn.tree import export_graphviz

# Export au format dot
export_graphviz(
  model.estimators_[0],
  out_file="tree.dot",
  rounded=True,
  proportion=False,
  precision=2,
  filled=True,
)

# Conversion en png avec une commande système (nécessite Graphviz)
from subprocess import call

call(["dot", "-Tpng", "tree.dot", "-o", "tree.png", "-Gdpi=600"])
# Conversion en svg
call(["dot", "-Tsvg", "tree.dot", "-o", "tree.svg"])

# Affichage dans le notebook
from IPython.display import Image

Image(filename="tree.png")

In [ ]:
from IPython.display import SVG

SVG(filename="tree.svg")